## Préparation traçable des données

Le CSV brut reste intact. `fiabiliser_nominations.py` crée les fichiers du dossier `fiabilisation/` à chaque exécution, en conservant les dictionnaires déjà révisés.

- `nominations_annotees.csv` : toutes les lignes sources, identifiants, alertes et provenance.
- `nominations_dedoublonnees.csv` : copie descriptive 2015–2025 ; seules les répétitions de même événement et même extrait normalisé sont écartées. Les autres ambiguïtés restent signalées.
- `referentiel_personnes.csv` et `identites_a_verifier.csv` : identités cohérentes ou provisoires, conflits et candidats. Aucun rapprochement approximatif automatique. Les identités sont des propositions, pas une certification documentaire.
- `doublons_audites.csv` et `nominations_simultanees.csv` : répétitions et événements de même personne/date à examiner. La simultanéité reste une hypothèse.
- `dictionnaires/ministeres.csv` : normalisation lexicale ; périmètres institutionnels distincts conservés. Les domaines sont des propositions multivaluées.
- `dictionnaires/postes.csv` : intitulé complet, fonction et famille proposées, niveau et nature de nomination. Les familles ne servent jamais à dédoublonner les postes.
- `dates_a_verifier.csv` : discordances avec les noms des PDF, sans correction automatique.

### Réviser les dictionnaires et les identités

Modifier les libellés et catégories dans les dictionnaires, conserver `valeur_originale` comme clé, ajouter une justification et une référence documentaire. Marquer `statut_validation=valide_manuellement` pour une validation documentée ; mettre à jour séparément `statut_codage_fonction` ou `statut_codage_domaine` après révision des catégories. Une absence d'alerte ne prouve pas l'exactitude du contenu.

Pour valider une identité, ajouter dans `dictionnaires/decisions_identites.csv` une ligne par observation concernée : `id_ligne_source`, `id_personne_valide`, `justification`, `reference_documentaire`. Utiliser un identifiant cible commun pour les observations réellement rattachées à la même personne ; examiner toutes les lignes d'un groupe en conflit. Relancer le traitement ; les décisions et les modifications de dictionnaire sont conservées.

### Limites de l'analyse de trajectoires existante

La partie II utilise un sous-ensemble conservateur : si une personne a un événement ambigu dans la période, sa trajectoire entière est exclue pour éviter des transitions artificielles. Les nominations sans matricule restent provisoires. Cette sélection peut modifier la population étudiée : comparer les effectifs et ne pas généraliser au corpus entier.

La visualisation par états ci-dessous suppose une continuité entre nominations. Les dates de fin de poste ne sont pas connues ; les intervalles sont des délais entre événements observés. Les familles proposées et le choix de 20 clusters restent à valider scientifiquement. Les mois sans source ne doivent pas être interprétés automatiquement comme des zéros.


# PARTIE I

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
#from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima.model import ARIMA

In [ ]:
# Base préparée commune à toutes les analyses ; CSV brut conservé intact.
from pathlib import Path
import json
from fiabiliser_nominations import run

BASE_DIR = Path.cwd()
if not (BASE_DIR / "nominations_2015_2025.csv").exists():
    raise FileNotFoundError("Ouvrir le notebook depuis le dossier contenant le CSV source.")
DOSSIER_FIABILISATION = BASE_DIR / "fiabilisation"
bilan_qualite = run(BASE_DIR / "nominations_2015_2025.csv", DOSSIER_FIABILISATION)
df_annote = pd.read_csv(DOSSIER_FIABILISATION / "nominations_annotees.csv", dtype=str, keep_default_na=False)
df = pd.read_csv(DOSSIER_FIABILISATION / "nominations_dedoublonnees.csv", dtype=str, keep_default_na=False)
print("Lignes sources :", len(df_annote), "| événements conservés 2015–2025 :", len(df))
df.head()


In [ ]:
# Le pipeline conserve toutes les sources dans nominations_annotees.csv.
# La copie descriptive est limitée à 2015–2025, sans poste vide, et écarte
# uniquement les répétitions textuelles documentées. Les autres alertes restent visibles.
assert df["dans_periode"].eq("1").all()
assert df["conserver_evenement"].eq("1").all()
df_annote.groupby(["statut_identite", "statut_doublon"], dropna=False).size().rename("nombre_lignes")


In [ ]:
# Les identifiants sont déterministes ; un identifiant provisoire n'est pas
# une personne certifiée. Le référentiel distingue les niveaux de confiance.
print("Identifiants dans la copie descriptive :", df["id_personne"].nunique())
print("Lignes éligibles aux trajectoires strictes :", bilan_qualite["nombre_lignes_trajectoires_strictes"])
print("Groupes de simultanéité à examiner :", bilan_qualite["nombre_groupes_simultanes"])
print("Ministères :", bilan_qualite["ministeres_originaux"], "→", bilan_qualite["ministeres_harmonises"])
print("Postes complets :", bilan_qualite["postes_originaux"], "→", bilan_qualite["postes_harmonises"])
pd.DataFrame({"limites": bilan_qualite["limites"]})


In [ ]:
# Copie du DataFrame
data = df.copy()

# Conversion en datetime
data["date_conseil"] = pd.to_datetime(
    data["date_conseil"], errors="coerce"
)

# Supprimer les dates invalides
data = data.dropna(subset=["date_conseil"])

# Trier chronologiquement
data = data.sort_values("date_conseil")

# Créer une série mensuelle
serie = (
    data
    .set_index("date_conseil")
    .resample("ME")
    .size()
)

print(serie.head(10))

In [ ]:
#Visualisation de la série temporelle
plt.figure(figsize=(14, 5))

plt.plot(serie)

plt.title("Évolution mensuelle du nombre de nominations")
plt.xlabel("Date")
plt.ylabel("Nombre de nominations")
plt.grid(True)

plt.show()

In [ ]:
# Moyenne  sur 12 mois
serie_ma12 = serie.rolling(window=12).mean()

plt.figure(figsize=(14, 5))

plt.plot(serie, label="Nombre de nominations")
plt.plot(serie_ma12, label="Moyenne mobile 12 mois")

plt.title("Évolution et tendance du nombre de nominations")
plt.xlabel("Date")
plt.ylabel("Nombre de nominations")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# Interprétation à reprendre après audit de couverture et réexécution sur la base préparée.


In [ ]:
# Regrouper les nominations par année
nominations_annuelles = (
    data
    .set_index("date_conseil")
    .resample("YE")
    .size()
)

print(nominations_annuelles)

In [ ]:
#Puis visualiser les nominations annuelles
nominations_annuelles.plot(
    kind="bar",
    figsize=(12, 5)
)

plt.title("Nombre annuel de nominations")
plt.xlabel("Année")
plt.ylabel("Nombre de nominations")

plt.show()

#### Analyse de la saisonnalite

In [ ]:
data["mois"] = data["date_conseil"].dt.month

moyenne_mensuelle = (
    data.groupby("mois")
    .size()
)

print(moyenne_mensuelle)

In [ ]:
moyenne_mensuelle = (
    serie
    .groupby(serie.index.month)
    .mean()
)

print(moyenne_mensuelle)

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    moyenne_mensuelle.index,
    moyenne_mensuelle.values,
    marker="o"
)

plt.xticks(range(1, 13))

plt.title("Nombre moyen de nominations par mois")
plt.xlabel("Mois")
plt.ylabel("Nombre moyen de nominations")

plt.grid(True)
plt.show()

In [ ]:
decomposition = seasonal_decompose(
    serie,
    model="additive",
    period=12
)

decomposition.plot()

plt.show()

In [ ]:
# Interprétation à reprendre après audit de couverture et réexécution sur la base préparée.


In [ ]:
# Verification de la stationnarité avec le test ADF
resultat_adf = adfuller(serie.dropna())

print("Statistique ADF :", resultat_adf[0])
print("p-value :", resultat_adf[1])

Résultat ADF à réévaluer sur la série préparée ; le test ne valide pas la couverture documentaire.

In [ ]:
# Test de KPSS
kpss_test = kpss(serie, regression='c', nlags=12)
print("KPSS p-value :", kpss_test[1])  

In [ ]:
# Interprétation à reprendre après audit de couverture et réexécution sur la base préparée.


In [ ]:
### Etudions les dependances temporelles

In [ ]:
fig = plt.figure(figsize=(12, 8))

plot_acf(
    serie,
    lags=12,
    ax=fig.add_subplot(211)
)

plot_pacf(
    serie,
    lags=12,
    ax=fig.add_subplot(212)
)

plt.tight_layout()
plt.show()

In [ ]:
# Interprétation à reprendre après audit de couverture et réexécution sur la base préparée.


In [ ]:
#Detection des anomalies dans la série temporelle
residus = decomposition.resid.dropna()

seuil = 3 * residus.std()

anomalies = residus[
    abs(residus) > seuil
]

print(anomalies)

In [ ]:
# Interprétation à reprendre après audit de couverture et réexécution sur la base préparée.


In [ ]:
# Prevision avec ARIMA
#model = ARIMA(
#    serie,
#    order=(0, 0, 2),
#)

#resultat = model.fit()

#print(resultat.summary())

# PARTIE II
# 1.	Représenter facilement plusieurs séquences temporelles
# 2.	Découvrir des trajectoires similaires
# 3.	Découvrir des motifs temporels
# 4.	Filtrer des populations selon des critères temporels
# 5.	Exploiter les trajectoires avec du Machine Learning

In [ ]:
# Liste des importations nécessaires

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import random
from datetime import datetime
import polars as pl
from collections import Counter

#Data simulation
#from tanat.dataset.simulation import (
#    generate_event_sequences,
#    generate_state_sequences,
#    generate_interval_sequences,
#)

# Sequence pools
from tanat.sequence import (
    EventSequencePool,
    StateSequencePool,
    IntervalSequencePool,
    EventSequenceSettings,
    StateSequenceSettings
)

from tanat.visualization.sequence import SequenceVisualizer

# Criterion for filtering
from tanat.criterion import (
    EntityCriterion,
    PatternCriterion,
    TimeCriterion,
    LengthCriterion,
    StaticCriterion,
)

from tanat.metric.entity import HammingEntityMetric
from tanat.metric.sequence import EditSequenceMetric
from tanat.metric.sequence import LinearPairwiseSequenceMetric
from tanat.clustering import HierarchicalClusterer

In [ ]:
# Même préparation qu'en partie I. On exclut toute la trajectoire d'une
# personne ayant un événement ambigu, pour éviter de créer un faux saut.
# Ce sous-ensemble conservateur sert à la sensibilité, pas à représenter le corpus entier.
data = pd.read_csv(
    DOSSIER_FIABILISATION / "nominations_trajectoires_strictes.csv",
    dtype=str, keep_default_na=False,
)
data["date_conseil"] = pd.to_datetime(data["date_conseil"], errors="raise")
print("Personnes avant filtrage de longueur :", data["id_personne"].nunique())
data.head()


In [ ]:
# Trois événements sur trois dates distinctes ; aucune succession intra-journalière inventée.
NB_OCC_MIN = 3
counts_by_id = data.groupby("id_personne")["date_conseil"].nunique()
id_min_occ = counts_by_id[counts_by_id >= NB_OCC_MIN].index.tolist()
data = data[data["id_personne"].isin(id_min_occ)].copy()
assert not data.duplicated(["id_personne", "date_conseil"]).any()
if data.empty:
    raise ValueError("Aucune séquence éligible : examiner le bilan et les fichiers de vérification.")
print(f"Personnes avec au moins {NB_OCC_MIN} dates : {len(id_min_occ)}")
print(f"Dimensions : {data.shape}")


In [ ]:
# 1 Représenter facilement plusieurs séquences temporelles
# Creation d'une base de données de séquences d'événements

In [ ]:
# Représentation exploratoire par états sous hypothèse de continuité.
# end_date est une borne jusqu'à l'événement suivant, jamais une fin de poste prouvée.
data = data.sort_values(["id_personne", "date_conseil"]).reset_index(drop=True)
data["end_date"] = data.groupby("id_personne")["date_conseil"].shift(-1)
# La dernière date du corpus n'établit pas que le suivi est exhaustif jusque-là.
FIN_OBSERVATION = pd.to_datetime(df["date_conseil"], errors="raise").max()
data["intervalle_censure"] = data["end_date"].isna()
data["end_date"] = data["end_date"].fillna(FIN_OBSERVATION)
data["fonction_harmonisee"] = pd.Categorical(data["fonction_harmonisee"])
assert (data["end_date"] >= data["date_conseil"]).all()
data.head(10)


In [ ]:
# Un stockage nouveau par exécution empêche de relire un ancien pool.
from tempfile import mkdtemp
builder = StateSequencePool.builder(end_value=FIN_OBSERVATION)
builder.add_dataframe(
    data, id_column="id_personne",
    features=["ministere_harmonise", "fonction_harmonisee"],
    start_column="date_conseil", end_column="end_date",
)
DOSSIER_FIABILISATION.mkdir(exist_ok=True)
store_path = Path(mkdtemp(prefix="tanat_", dir=DOSSIER_FIABILISATION)) / "store"
store = builder.build(store_path=str(store_path))


In [ ]:
# Création du pool de séquences d'états
sequence_pool = StateSequencePool(
    store,
    id_column="id_personne",
    start_column="date_conseil",
    end_column="end_date",
    entity_features=["ministere_harmonise", "fonction_harmonisee"]
)

In [ ]:
print(sequence_pool)
assert set(sequence_pool.unique_ids) == set(data["id_personne"]), "Le pool ne correspond pas aux données préparées."


In [ ]:
# Affiche les informations sur le store
sequence_pool.describe()

In [ ]:
# Tirage reproductible parmi les personnes, sans surpondérer les longues séquences.
id_personne = random.Random(42).choice(sorted(sequence_pool.unique_ids))
person_sequence = sequence_pool[id_personne]
print(f"Personne {id_personne} : {len(person_sequence)} événements observés")
SequenceVisualizer.timeline(group_by="id", time_mode='absolute').title(
    f"Personne {id_personne} — continuité entre événements supposée"
).draw(person_sequence, entity_feature="fonction_harmonisee")


In [ ]:
frequences_ministeres = (
    sequence_pool
    .temporal_data(features=["ministere_harmonise"])
    ["ministere_harmonise"]
    .value_counts()
    .head(20)
    .rename_axis("ministere_harmonise")
    .reset_index(name="nombre")
)

frequences_ministeres

In [ ]:
sns.barplot(data=frequences_ministeres, x="nombre", y="ministere_harmonise")

In [ ]:
# 2. Découvrir des trajectoires similaires
# Il existe des métriques pour comparer les états, les séquences et les trajectoires.
# Dans chacun de ces cas, il y a plusieurs métriques possibles.

In [ ]:
# Metriques pour comparer les séquences d'états
# la metrique de distance d'edition (Edit Distance) est une mesure de similarité entre deux séquences. 
# Elle est définie comme le nombre minimum d'opérations nécessaires pour transformer une séquence en une autre.
# Les opérations autorisées sont généralement l'insertion, la suppression et la substitution d'un élément.
hamming = HammingEntityMetric(entity_feature="fonction_harmonisee")
metric1 = EditSequenceMetric(entity_metric=hamming, indel_cost=1.0, normalize=True)
print(metric1)

In [ ]:
ids = sequence_pool.unique_ids
dist = metric1(sequence_pool[ids[0]], sequence_pool[ids[1]])
print(f"Distance between {ids[0]} and {ids[1]}: {dist:.4f}")
# le est 1. Ce qui signifie que les deux séquences sont très différentes, car elles n'ont pas de fonction en commun.

In [ ]:
# Calcul de la matrice de distance pour toutes les séquences
dm = metric1.compute_matrix(sequence_pool)
print(f"Distance matrix shape: {dm.shape}")

In [ ]:
arr = dm.to_numpy()
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(arr, cmap="viridis_r", vmin=0, vmax=1)
ax.set_title("Matrice de distance d'edition", fontsize=12, fontweight="bold")
ax.set_xlabel("Sequence")
ax.set_ylabel("Sequence")
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Distance")
plt.tight_layout()
plt.show()

In [ ]:
# Il serait preferable d'avoir des groupes de sequences...
# Mais si c'est un groupe, comment va t il calculer les distences?

In [ ]:
# la metrique LinearPairwiseSequenceMetric
metric2 = LinearPairwiseSequenceMetric(
    entity_metric=hamming, agg_fun="mean", padding_penalty=1.0
)
print(metric2)

In [ ]:
# Calculons la distance entre une seule paire
ids = sequence_pool.unique_ids
dist = metric2(sequence_pool[ids[0]], sequence_pool[ids[1]])
print(f"Distance between {ids[0]} and {ids[1]}: {dist:.4f}")


In [ ]:
# Appliquer a toutes les sequences
dm = metric2.compute_matrix(sequence_pool)
print(f"Distance matrix shape: {dm.shape}")

In [ ]:
arr = dm.to_numpy()
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(arr, cmap="viridis_r", vmin=0, vmax=1)
ax.set_title("Matrice de distance LinearPairwise", fontsize=12, fontweight="bold")
ax.set_xlabel("Sequence")
ax.set_ylabel("Sequence")
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Distance")
plt.tight_layout()
plt.show()

In [ ]:
# Classifions les sequences
# classification hierarchique
clusterer = HierarchicalClusterer(
    metric=metric1,
    n_clusters=20,
)

clusterer.fit(sequence_pool)

In [ ]:
print(clusterer.clusters)

In [ ]:
print(sequence_pool.static_data().head())

In [ ]:
# 3 Découvrir des motifs temporels

In [ ]:
# Récupérer les séquences
seqs = sequence_pool.get_sequences()
# Récupérer les clusters
static = sequence_pool.static_data()
# Construire les groupes d'ID
groupes = {
    cluster_id: groupe["id_personne"].tolist()
    for cluster_id, groupe in static.groupby("__HCLUSTERS__")
}
# Associer chaque ID à son objet Sequence
groupes_sequences = {
    cluster_id: {
        id_personne: seqs[id_personne]
        for id_personne in ids_cluster
    }
    for cluster_id, ids_cluster in groupes.items()
}
groupes_sequences

In [ ]:
nb_total = sum(len(groupe) for groupe in groupes_sequences.values())

print("Nombre total de séquences :", nb_total)
print("Nombre de séquences dans pool :", len(seqs))

In [ ]:
# Recuperons les donnees du premier cluster
cluster_0 = groupes_sequences[0]
print("Nombre de personnes dans le cluster 0 :", len(cluster_0))

In [ ]:
# Transformons les sequences du cluster 0 en dataframe

donnees_cluster_0 = []

for id_personne, sequence in cluster_0.items():

    df_seq = sequence.temporal_data().copy()

    donnees_cluster_0.append(df_seq)

cluster_0_df = pd.concat(
    donnees_cluster_0,
    ignore_index=True
)
print(cluster_0_df.shape)
print(cluster_0_df.head())

In [ ]:
# Quelles fonction domine dans le premier cluster

fonctions = (
    cluster_0_df["fonction_harmonisee"]
    .value_counts()
)

print(fonctions.head(20))

In [ ]:
# Les fonctions les plus représentées parmi les positions observées dans le cluster 0 en %
pourcentage_fonctions = (
    cluster_0_df["fonction_harmonisee"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(pourcentage_fonctions.head(20))

In [ ]:
# Intervalles jusqu'à la nomination suivante, sous hypothèse de suivi complet.
# Exclure les derniers épisodes de ce tableau : leur délai est censuré.
# Ce résumé des intervalles complets n'est pas un estimateur de durée en poste.
cluster_0_df = cluster_0_df.sort_values(["id_personne", "date_conseil"]).copy()
cluster_0_df["prochaine_nomination_observee"] = cluster_0_df.groupby("id_personne")["date_conseil"].shift(-1)
cluster_0_df["duree"] = cluster_0_df["prochaine_nomination_observee"] - cluster_0_df["date_conseil"]


In [ ]:
cluster_0_df["duree_jours"] = (
    cluster_0_df["duree"]
    .dt.total_seconds()
    / (24 * 3600)
)

In [ ]:
cluster_0_df["duree_annees"] = (
    cluster_0_df["duree_jours"] / 365.25
)

In [ ]:
duree_par_fonction = (
    cluster_0_df
    .groupby("fonction_harmonisee")["duree_annees"]
    .agg(
        nombre="count",
        duree_totale="sum",
        duree_moyenne="mean",
        duree_mediane="median"
    )
    .sort_values(
        "duree_totale",
        ascending=False
    )
)

print(duree_par_fonction.head(20))

In [ ]:
# Trier les séquences
cluster_0_df = cluster_0_df.sort_values(
    ["id_personne", "date_conseil"]
).reset_index(drop=True)

# Fonction suivante
cluster_0_df["fonction_suivante"] = (
    cluster_0_df
    .groupby("id_personne")["fonction_harmonisee"]
    .shift(-1)
)

# Supprimer les dernières positions
transitions_df = cluster_0_df.dropna(
    subset=["fonction_suivante"]
).copy()

# Convertir les colonnes PyArrow dictionary en chaînes
transitions_df["fonction_harmonisee"] = (
    transitions_df["fonction_harmonisee"]
)

transitions_df["fonction_suivante"] = (
    transitions_df["fonction_suivante"].astype(str)
)

# Garder uniquement les changements de fonction
transitions_df = transitions_df[
    transitions_df["fonction_harmonisee"]
    != transitions_df["fonction_suivante"]
].copy()

# Construire la transition
transitions_df["transition"] = (
    transitions_df["fonction_harmonisee"]
    + " → "
    + transitions_df["fonction_suivante"]
)

# Fréquence des transitions
transitions_reelles = (
    transitions_df["transition"]
    .value_counts()
)

print(transitions_reelles.head(20))